In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv("./data/StudentsPerformance.csv")
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   gender                       1000 non-null   str  
 1   race/ethnicity               1000 non-null   str  
 2   parental level of education  1000 non-null   str  
 3   lunch                        1000 non-null   str  
 4   test preparation course      1000 non-null   str  
 5   math score                   1000 non-null   int64
 6   reading score                1000 non-null   int64
 7   writing score                1000 non-null   int64
dtypes: int64(3), str(5)
memory usage: 62.6 KB


In [27]:
df_cols= df.columns
df_cols

Index(['gender', 'race/ethnicity', 'parental level of education', 'lunch',
       'test preparation course', 'math score', 'reading score',
       'writing score'],
      dtype='str')

In [28]:
df.columns = [col.strip().lower().replace(" ", "_").replace("/", "_") for col in df_cols]
df.columns

Index(['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch',
       'test_preparation_course', 'math_score', 'reading_score',
       'writing_score'],
      dtype='str')

In [29]:
str_cols = df.select_dtypes(include="str").columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
df["lunch"] = df["lunch"].replace("/", "_")
df["race_ethnicity"] = df["race_ethnicity"].replace(" ", "_")

In [30]:
missing = df.isnull().sum()
print("Missing Values per column:\n", missing[missing > 0] if missing.sum() else "None Found")
print(f"Dublicate rows found: {df.duplicated().sum()}")

Missing Values per column:
 None Found
Dublicate rows found: 0


In [31]:
score_cols = ['math_score', 'reading_score', 'writing_score']
for col in score_cols:
    invalid = df[(df[col] < 0) | (df[col] > 100)]
    if(len(invalid)):
        print(f"There is {len(invalid)} invalid values in {col}")


In [ ]:
for col in score_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    IQR = q3 - q1
    lower_bound, upper_bound = q1 - 1.5 * IQR, q3 + 1.5 * IQR
    n_outlier = ((df[col] < lower_bound)|(df[col] > upper_bound)).sum()
    print(f"the number of outliers in {col} column is {n_outlier}")


the number of outliers in math_score column is 8
the number of outliers in reading_score column is 6
the number of outliers in writing_score column is 5


In [33]:
df["total_score"] = df[score_cols].sum(axis=1)
df["average_score"] = df[score_cols].mean(axis=1).round(2)

pass_mark = 60
for c in score_cols:
    df[f"{c.split('_')[0]}_pass"] = df[c] >= pass_mark

df["all_subjects_passed"] = df[[f"{c.split('_')[0]}_pass" for c in score_cols]].all(axis=1)

In [34]:
def grade(avg):
    if avg >= 90: return "A"
    if avg >= 80: return "B"
    if avg >= 65: return "C"
    if avg >= 50: return "D"
    return "F"

In [35]:
df["grade"] = df["average_score"].apply(grade).astype("category")

df["score_std"] = df[score_cols].std(axis=1).round(2)
df["score_range"] = (df[score_cols].max(axis=1) - df[score_cols].min(axis=1))

In [36]:
subj_map = {"math_score": "math", "reading_score": "reading", "writing_score": "writing"}
df["strongest_subject"] = df[score_cols].idxmax(axis=1).map(subj_map).astype("category")
df["weakest_subject"] = df[score_cols].idxmin(axis=1).map(subj_map).astype("category")

In [37]:
print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"Columns:\n{list(df.columns)}")
print(f"Grade distribution:\n{df['grade'].value_counts().sort_index()}")
print(f"Pass rate (all subjects): {df['all_subjects_passed'].mean():.1%}")

FINAL DATASET SUMMARY
Shape: (1000, 19)
Columns:
['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course', 'math_score', 'reading_score', 'writing_score', 'total_score', 'average_score', 'math_pass', 'reading_pass', 'writing_pass', 'all_subjects_passed', 'grade', 'score_std', 'score_range', 'strongest_subject', 'weakest_subject']
Grade distribution:
grade
A     52
B    146
C    403
D    296
F    103
Name: count, dtype: int64
Pass rate (all subjects): 60.3%


In [ ]:
df.to_csv("./data/student_performance_cleaned.csv", index=False)
print("Saved cleaned dataset")

Saved cleaned dataset
